In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
log_df = pd.read_csv("log.csv")
trace_df = pd.read_csv("trace.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
for df in (metric_df, log_df, trace_df, error_df):
    # keep original column name 'timestamp' and also add parsed 'ts' for convenience
    df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Helper to build summaries
def build_summary(df, kpi_col=None, msg_col=None):
    total_rows = len(df)
    time_min = df['ts'].min()
    time_max = df['ts'].max()
    cmdb_counts = df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count')
    overview = pd.DataFrame([{
        'total_rows': total_rows,
        'unique_cmdb_count': cmdb_counts.shape[0],
        'time_min_utc': time_min,
        'time_max_utc': time_max,
        'unique_kpi_or_msg_count': df[kpi_col].nunique() if kpi_col else (df[msg_col].nunique() if msg_col else 0)
    }])
    if kpi_col:
        kpi_counts = df[kpi_col].value_counts().rename_axis(kpi_col).reset_index(name='count')
        kpi_counts_top = kpi_counts.head(20)
    elif msg_col:
        kpi_counts = df[msg_col].value_counts().rename_axis(msg_col).reset_index(name='count')
        kpi_counts_top = kpi_counts.head(20)
    else:
        kpi_counts = pd.DataFrame()
        kpi_counts_top = pd.DataFrame()
    return overview, cmdb_counts, kpi_counts_top

# Build summaries for each file
metric_overview, metric_cmdb_counts, metric_kpi_counts = build_summary(metric_df, kpi_col='kpi_name')
log_overview, log_cmdb_counts, log_kpi_counts = build_summary(log_df, kpi_col='log_name')
trace_overview, trace_cmdb_counts, trace_kpi_counts = build_summary(trace_df, kpi_col='trace_name')
error_overview, error_cmdb_counts, error_message_counts = build_summary(error_df, msg_col='message')

# Keep outputs compact: show overviews and top counts (cmdb counts usually small)
metric_overview, metric_cmdb_counts, metric_kpi_counts, \
log_overview, log_cmdb_counts, log_kpi_counts, \
trace_overview, trace_cmdb_counts, trace_kpi_counts, \
error_overview, error_cmdb_counts, error_message_counts

```
Out[1]:
```
```python
# Summarize previous execution results into a concise plain-English string.
metric_top = metric_cmdb_counts.head(6).apply(lambda r: f"{r['cmdb_id']}({r['count']})", axis=1).tolist()
metric_kpis = metric_kpi_counts.set_index('kpi_name')['count'].to_dict()

log_top = log_cmdb_counts.head(6).apply(lambda r: f"{r['cmdb_id']}({r['count']})", axis=1).tolist()
log_kpis = log_kpi_counts.set_index('log_name')['count'].to_dict()

summary = (
    f"metric.csv: {int(metric_overview.total_rows.iloc[0])} rows across "
    f"{int(metric_overview.unique_cmdb_count.iloc[0])} cmdb_ids; time range "
    f"{metric_overview.time_min_utc.iloc[0]} — {metric_overview.time_max_utc.iloc[0]} UTC. "
    f"Top cmdb_ids: {', '.join(metric_top)}. "
    f"{int(metric_overview.unique_kpi_or_msg_count.iloc[0])} unique KPI names (top counts: "
    f"{', '.join([f'{k} {v}' for k,v in metric_kpis.items()])}).\n\n"
    f"log.csv: {int(log_overview.total_rows.iloc[0])} rows across "
    f"{int(log_overview.unique_cmdb_count.iloc[0])} cmdb_ids; time range "
    f"{log_overview.time_min_utc.iloc[0]} — {log_overview.time_max_utc.iloc[0]} UTC. "
    f"Top cmdb_ids: {', '.join(log_top)}. "
    f"{int(log_overview.unique_kpi_or_msg_count.iloc[0])} unique log names (counts: "
    f"{', '.join([f'{k} {v}' for k,v in log_kpis.items()])}).\n\n"
    f"trace.csv: {int(trace_overview.total_rows.iloc[0])} rows (no trace data available).\n\n"
    f"error_logs.csv: {int(error_overview.total_rows.iloc[0])} rows (no error log data available)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows  unique_cmdb_count              time_min_utc              time_max_utc  unique_kpi_or_msg_count
0        1974                 15 2024-01-19 03:50:00+00:00 2024-01-19 04:14:00+00:00                        8,               cmdb_id  count
0           catalogue    186
1           front-end    175
2               carts    173
3              orders    173
4                user    173
5             payment    171
6            shipping    148
7           orders-db    100
8        catalogue-db    100
9            carts-db    100
10       queue-master    100
11         session-db    100
12           rabbitmq    100
13            user-db    100
14  rabbitmq-exporter     75,      kpi_name  count
0         cpu    375
1         mem    375
2      socket    375
3      diskio    188
4    workload    175
5  latency-50    168
6  latency-90    168
7       error    150,    total_rows  unique_cmdb_count              time_min_utc              time_max_utc  unique_kpi_or_msg_count
0         436                 10 2024-01-19 03:50:00+00:00 2024-01-19 04:14:00+00:00                        2,         cmdb_id  count
0     front-end     50
1       payment     50
2          user     50
3      carts-db     48
4     catalogue     48
5        orders     46
6      shipping     46
7  queue-master     46
8         carts     42
9    session-db     10,           log_name  count
0  log.error_count    218
1    log.row_count    218,    total_rows  unique_cmdb_count time_min_utc time_max_utc  unique_kpi_or_msg_count
0           0                  0          NaT          NaT                        0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [trace_name, count]
Index: [],    total_rows  unique_cmdb_count time_min_utc time_max_utc  unique_kpi_or_msg_count
0           0                  0          NaT          NaT                        0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [message, count]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# reuse metric_df if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# ensure timestamp parsed to UTC datetime column 'ts'
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Candidate components
candidates = ['carts', 'catalogue', 'orders', 'payment', 'user']

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full series (no time filter)
p95_df = (
    metric_df[metric_df['cmdb_id'].isin(candidates)]
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95')
)

# 2) Filter to incident window (UTC)
start = pd.Timestamp("2024-01-19 03:47:24", tz='UTC')
end = pd.Timestamp("2024-01-19 04:17:24", tz='UTC')
window_df = metric_df[
    (metric_df['cmdb_id'].isin(candidates)) &
    (metric_df['ts'] >= start) &
    (metric_df['ts'] <= end)
].copy()

# Merge p95 into window rows for anomaly detection (only rows present in window)
window_with_p95 = pd.merge(
    window_df,
    p95_df,
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# mark anomalies where value >= p95 (p95 should exist for these combos)
window_with_p95['is_anomaly'] = window_with_p95['value'] >= window_with_p95['p95']

# group window metrics
window_agg = (
    window_with_p95
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        total_rows_in_window=('value', 'size'),
        anomaly_count_in_window=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp_utc=('ts', lambda s: s[window_with_p95.loc[s.index, 'is_anomaly']].min() if len(s) > 0 else pd.NaT),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)

# 2b) Some combos may have no rows in-window; include them with zeros/NaN
result = pd.merge(p95_df, window_agg, on=['cmdb_id', 'kpi_name'], how='left')

# fill defaults for combos with no window data
result['total_rows_in_window'] = result['total_rows_in_window'].fillna(0).astype(int)
result['anomaly_count_in_window'] = result['anomaly_count_in_window'].fillna(0).astype(int)
# earliest_anomaly_timestamp_utc remains NaT if none
# max_value_in_window stays NaN if none

# compute breach_ratio = (max_value - p95)/p95, NaN if p95==0 or max_value NaN
result['breach_ratio'] = (result['max_value_in_window'] - result['p95']) / result['p95']
result.loc[result['p95'] == 0, 'breach_ratio'] = np.nan

# ensure earliest_anomaly_timestamp_utc is UTC tz-aware or NaT
# (it should already be tz-aware from 'ts', but ensure dtype)
result['earliest_anomaly_timestamp_utc'] = pd.to_datetime(result['earliest_anomaly_timestamp_utc']).dt.tz_convert('UTC')

# select and order columns as requested
result = result[
    ['cmdb_id', 'kpi_name', 'p95', 'total_rows_in_window', 'anomaly_count_in_window',
     'earliest_anomaly_timestamp_utc', 'max_value_in_window', 'breach_ratio']
]

# sort by anomaly_count desc then breach_ratio desc
result = result.sort_values(by=['anomaly_count_in_window', 'breach_ratio'], ascending=[False, False])

# keep output compact: top 20 rows
result_head = result.head(20)

result_head

```
Out[2]:
```
```python
# Summarize the result DataFrame from the previous analysis into a concise plain-English string.
# Reuse the existing `result` DataFrame (from prior execution).

try:
    df = result.copy()
except NameError:
    import pandas as pd
    df = pd.read_csv("metric.csv")  # fallback (should not be needed)

# Top groups with full-window anomalies (anomaly_count == max observed)
full_anom = df[df['anomaly_count_in_window'] == df['anomaly_count_in_window'].max()]
full_list = []
for _, r in full_anom.iterrows():
    full_list.append(f"{r['cmdb_id']}/{r['kpi_name']} (p95={r['p95']}, anomalies={int(r['anomaly_count_in_window'])}, earliest={r['earliest_anomaly_timestamp_utc']})")

# Top breach ratios among groups with anomalies (exclude NaN breach_ratio)
breach_candidates = df[(df['anomaly_count_in_window']>0) & (df['breach_ratio'].notna())].sort_values('breach_ratio', ascending=False).head(5)
breach_list = []
for _, r in breach_candidates.iterrows():
    breach_list.append(f"{r['cmdb_id']}/{r['kpi_name']} (p95={r['p95']}, max={r['max_value_in_window']}, breach_ratio={r['breach_ratio']:.3f}, anomalies={int(r['anomaly_count_in_window'])})")

summary = (
    "Summary of metric anomalies in the incident window (2024-01-19 03:47:24 to 2024-01-19 04:17:24 UTC):\n\n"
    f"1) Groups with the highest anomaly counts (all rows in-window flagged):\n   - " + "; ".join(full_list) + "\n\n"
    "Note: several 'error' KPIs have p95 == 0, so value >= p95 caused all in-window rows to be counted as anomalies.\n\n"
    "2) Groups with the largest relative breaches (top examples):\n   - " + ("; ".join(breach_list) if breach_list else "None with numeric breach_ratio available") + "\n\n"
    "3) Observations:\n   - payment/socket shows 25/25 rows anomalous with p95=5.0 (earliest anomaly 2024-01-19 03:50:00+00:00).\n   - Many services (carts, catalogue, orders, payment, user) show 'error' KPI anomaly counts of 25 with p95=0.\n   - orders latency-50 and latency-90 show the largest relative breaches (breach_ratio ~6.60 and ~2.11 respectively), but only 2 anomaly rows each.\n\n"
    "Overall: multiple components show frequent anomalies in-window; the 'error' KPI results are driven by p95==0 behavior, while latency KPIs (orders) show the most significant relative increases."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name        p95  total_rows_in_window  anomaly_count_in_window earliest_anomaly_timestamp_utc  max_value_in_window  breach_ratio
27    payment      socket   5.000000                    25                       25      2024-01-19 03:50:00+00:00             5.000000      0.000000
1       carts       error   0.000000                    25                       25      2024-01-19 03:50:00+00:00             0.000000           NaN
9   catalogue       error   0.000000                    25                       25      2024-01-19 03:50:00+00:00             0.000000           NaN
16     orders       error   0.000000                    25                       25      2024-01-19 03:50:00+00:00             0.000000           NaN
23    payment       error   0.000000                    25                       25      2024-01-19 03:50:00+00:00             0.000000           NaN
30       user       error   0.000000                    25                       25      2024-01-19 03:50:00+00:00             0.000000           NaN
34       user      socket  18.000000                    25                       14      2024-01-19 03:50:00+00:00            18.000000      0.000000
17     orders  latency-50   0.191025                    24                        2      2024-01-19 03:52:00+00:00             1.452307      6.602722
18     orders  latency-90   1.016053                    24                        2      2024-01-19 03:52:00+00:00             3.155017      2.105169
11  catalogue  latency-90   0.004955                    24                        2      2024-01-19 03:51:00+00:00             0.007750      0.564159
2       carts  latency-50   0.801530                    24                        2      2024-01-19 03:52:00+00:00             1.244652      0.552845
3       carts  latency-90   3.760871                    24                        2      2024-01-19 03:52:00+00:00             5.386613      0.432278
5       carts      socket  48.133333                    25                        2      2024-01-19 03:52:00+00:00            56.733333      0.178670
13  catalogue      socket   9.953333                    25                        2      2024-01-19 03:52:00+00:00            11.483333      0.153717
7   catalogue         cpu  21.091219                    25                        2      2024-01-19 04:03:00+00:00            23.356874      0.107422
10  catalogue  latency-50   0.003140                    24                        2      2024-01-19 03:51:00+00:00             0.003375      0.074934
24    payment  latency-50   0.002647                    23                        2      2024-01-19 03:52:00+00:00             0.002783      0.051025
25    payment  latency-90   0.004925                    23                        2      2024-01-19 04:01:00+00:00             0.005151      0.046037
20     orders      socket  18.776667                    25                        2      2024-01-19 03:55:00+00:00            19.583333      0.042961
29       user         cpu   1.046120                    25                        2      2024-01-19 04:11:00+00:00             1.090183      0.042120```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing log_df if available, else load
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")

# Ensure timestamp parsed to UTC datetime column 'ts'
if 'ts' not in log_df.columns:
    log_df['ts'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ['carts', 'catalogue', 'orders', 'payment', 'user']

# 1) Compute global P95 per (cmdb_id, log_name) using full series (only for candidate cmdb_ids)
p95_df = (
    log_df[log_df['cmdb_id'].isin(candidates)]
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95')
)

# 2) Filter to incident window (UTC)
start = pd.Timestamp("2024-01-19 03:47:24", tz='UTC')
end = pd.Timestamp("2024-01-19 04:17:24", tz='UTC')
window_df = log_df[
    (log_df['cmdb_id'].isin(candidates)) &
    (log_df['ts'] >= start) &
    (log_df['ts'] <= end)
].copy()

# Merge p95 into window rows for anomaly detection
window_with_p95 = pd.merge(
    window_df,
    p95_df,
    on=['cmdb_id', 'log_name'],
    how='right'  # keep combos from p95_df even if not present in-window (we'll handle)
)

# After right-merge, rows that came only from p95_df have NaNs in value/ts; keep them for completeness
# Determine anomalies where value >= p95 (only valid when value notna)
window_with_p95['is_anomaly'] = False
mask_valid = window_with_p95['value'].notna() & window_with_p95['p95'].notna()
window_with_p95.loc[mask_valid, 'is_anomaly'] = window_with_p95.loc[mask_valid, 'value'] >= window_with_p95.loc[mask_valid, 'p95']

# Aggregate window stats for entries that actually have timestamps (i.e., existed in-window)
in_window_only = window_with_p95[window_with_p95['ts'].notna()].copy()

total_rows = in_window_only.groupby(['cmdb_id', 'log_name'], dropna=False).size().reset_index(name='total_rows_in_window')
anomaly_counts = in_window_only.groupby(['cmdb_id', 'log_name'], dropna=False)['is_anomaly'].sum().reset_index(name='anomaly_count_in_window')
earliest_anom = in_window_only[in_window_only['is_anomaly']].groupby(['cmdb_id', 'log_name'], dropna=False)['ts'].min().reset_index(name='earliest_anomaly_timestamp_utc')
max_values = in_window_only.groupby(['cmdb_id', 'log_name'], dropna=False)['value'].max().reset_index(name='max_value_in_window')

# Merge aggregates with p95_df to ensure all combinations present in full series are included
log_result = p95_df.merge(total_rows, on=['cmdb_id', 'log_name'], how='left') \
                   .merge(anomaly_counts, on=['cmdb_id', 'log_name'], how='left') \
                   .merge(earliest_anom, on=['cmdb_id', 'log_name'], how='left') \
                   .merge(max_values, on=['cmdb_id', 'log_name'], how='left')

# Fill defaults for combos with no in-window data
log_result['total_rows_in_window'] = log_result['total_rows_in_window'].fillna(0).astype(int)
log_result['anomaly_count_in_window'] = log_result['anomaly_count_in_window'].fillna(0).astype(int)
# earliest_anomaly_timestamp_utc and max_value_in_window remain NaT/NaN if absent

# Compute breach_ratio = (max_value - p95)/p95, NaN if p95==0 or max_value missing
log_result['breach_ratio'] = (log_result['max_value_in_window'] - log_result['p95']) / log_result['p95']
log_result.loc[(log_result['p95'] == 0) | (log_result['max_value_in_window'].isna()), 'breach_ratio'] = np.nan

# Ensure earliest timestamp timezone is UTC (if present)
if 'earliest_anomaly_timestamp_utc' in log_result.columns:
    log_result['earliest_anomaly_timestamp_utc'] = pd.to_datetime(log_result['earliest_anomaly_timestamp_utc']).dt.tz_convert('UTC')

# Select and order columns as requested
log_result = log_result[
    ['cmdb_id', 'log_name', 'p95', 'total_rows_in_window', 'anomaly_count_in_window',
     'earliest_anomaly_timestamp_utc', 'max_value_in_window', 'breach_ratio']
]

# Sort by anomaly_count desc then breach_ratio desc
log_result = log_result.sort_values(by=['anomaly_count_in_window', 'breach_ratio'], ascending=[False, False])

# Keep output compact: top 20 rows
log_result_head = log_result.head(20)

log_result_head

```
Out[3]:
```
```python
# Build a concise plain-English summary from the previously computed `log_result_head`.
try:
    df = log_result_head.copy()
except NameError:
    # fallback: load and recompute minimal required info (unlikely needed)
    import pandas as pd, numpy as np
    log_df = pd.read_csv("log.csv")
    log_df['ts'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
    candidates = ['carts','catalogue','orders','payment','user']
    p95_df = (
        log_df[log_df['cmdb_id'].isin(candidates)]
        .groupby(['cmdb_id','log_name'])['value'].quantile(0.95).reset_index(name='p95')
    )
    start = pd.Timestamp("2024-01-19 03:47:24", tz='UTC')
    end = pd.Timestamp("2024-01-19 04:17:24", tz='UTC')
    window = log_df[(log_df['cmdb_id'].isin(candidates)) & (log_df['ts']>=start) & (log_df['ts']<=end)]
    merged = p95_df.merge(window.groupby(['cmdb_id','log_name']).size().reset_index(name='total_rows_in_window'),
                          on=['cmdb_id','log_name'], how='left')
    merged['total_rows_in_window'] = merged['total_rows_in_window'].fillna(0).astype(int)
    df = merged.head(20)

# Summarize error_count rows (p95==0 driven)
err = df[df['log_name']=='log.error_count']
err_lines = []
for _, r in err.iterrows():
    earliest = r['earliest_anomaly_timestamp_utc']
    err_lines.append(f"{r['cmdb_id']}: {int(r['anomaly_count_in_window'])}/{int(r['total_rows_in_window'])} rows anomalous (p95={r['p95']}, earliest={earliest})")

# Summarize log.row_count rows (small numeric breaches)
rowcount = df[df['log_name']=='log.row_count']
row_lines = []
for _, r in rowcount.iterrows():
    row_lines.append(f"{r['cmdb_id']}: anomalies={int(r['anomaly_count_in_window'])}, p95={r['p95']}, max={r['max_value_in_window']}, breach_ratio={r['breach_ratio']:.3f}")

summary = (
    "Summary of log.csv analysis for candidate services in the incident window (2024-01-19 03:47:24 — 04:17:24 UTC):\n\n"
    "1) log.error_count:\n  - " + "\n  - ".join(err_lines) + "\n\n"
    "   Observations: For payment, user, catalogue, orders, and carts the p95 for log.error_count is 0.0, "
    "so all in-window rows are flagged as anomalies (e.g., payment 25/25, user 25/25, catalogue 24/24, orders 23/23, carts 21/21). "
    "These anomaly counts are driven by p95==0 behavior rather than large numeric increases.\n\n"
    "2) log.row_count (numeric breaches):\n  - " + ("\n  - ".join(row_lines) if len(row_lines)>0 else "None") + "\n\n"
    "   Observations: A few services show small increases above p95 in log.row_count (small breach ratios: catalogue ~0.039, orders ~0.017, payment ~0.012, user ~0.003, carts ~0.000).\n\n"
    "Conclusion: The dominant signal in logs is log.error_count flagged as anomalous due to p95==0 across multiple services; log.row_count shows only minor numeric breaches."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         log_name    p95  total_rows_in_window  anomaly_count_in_window earliest_anomaly_timestamp_utc  max_value_in_window  breach_ratio
6    payment  log.error_count    0.0                    25                       25      2024-01-19 03:50:00+00:00                    0           NaN
8       user  log.error_count    0.0                    25                       25      2024-01-19 03:50:00+00:00                    0           NaN
2  catalogue  log.error_count    0.0                    24                       24      2024-01-19 03:51:00+00:00                    0           NaN
4     orders  log.error_count    0.0                    23                       23      2024-01-19 03:52:00+00:00                    0           NaN
0      carts  log.error_count    0.0                    21                       21      2024-01-19 03:52:00+00:00                    0           NaN
3  catalogue    log.row_count  162.7                    24                        2      2024-01-19 03:58:00+00:00                  169      0.038722
5     orders    log.row_count  127.8                    23                        2      2024-01-19 03:58:00+00:00                  130      0.017214
7    payment    log.row_count  103.8                    25                        2      2024-01-19 03:58:00+00:00                  105      0.011561
9       user    log.row_count  604.2                    25                        2      2024-01-19 03:58:00+00:00                  606      0.002979
1      carts    log.row_count  100.0                    21                        2      2024-01-19 03:58:00+00:00                  100      0.000000```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse metric_df if available, else load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Ensure ts column exists and is UTC timezone-aware
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Focus on cmdb_id == 'orders'
comp = 'orders'
orders_full = metric_df[metric_df['cmdb_id'] == comp].copy()

# 1) Compute global P95 per kpi_name using full series (no time filtering)
p95_series = orders_full.groupby('kpi_name')['value'].quantile(0.95)
p95_df = p95_series.reset_index().rename(columns={'value': 'p95'})

# Incident window (UTC)
start = pd.Timestamp("2024-01-19 03:47:24", tz='UTC')
end = pd.Timestamp("2024-01-19 04:17:24", tz='UTC')

# Filter orders rows to window
orders_window = orders_full[(orders_full['ts'] >= start) & (orders_full['ts'] <= end)].copy()

# Prepare list of all KPI names for orders (ensure include KPIs even if no rows in-window)
kpi_names = p95_df['kpi_name'].tolist()

rows = []
for kpi in kpi_names:
    p95_val = float(p95_df.loc[p95_df['kpi_name'] == kpi, 'p95'].iloc[0])
    kdf = orders_window[orders_window['kpi_name'] == kpi].copy().sort_values('ts')
    total_rows = int(len(kdf))
    if total_rows > 0:
        # mark anomalies
        kdf['is_anom'] = kdf['value'] >= p95_val
        anomaly_count = int(kdf['is_anom'].sum())
        max_value = float(kdf['value'].max())
        earliest_anom_ts = kdf.loc[kdf['is_anom'], 'ts'].min() if anomaly_count > 0 else pd.NaT
        breach_ratio = np.nan
        if (p95_val != 0) and (not np.isnan(max_value)):
            breach_ratio = (max_value - p95_val) / p95_val
        else:
            breach_ratio = np.nan

        # identify consecutive anomaly segments (consecutive minutes)
        # compute time diff and previous anomaly flag
        kdf = kdf.sort_values('ts').reset_index(drop=True)
        kdf['ts_diff'] = kdf['ts'].diff()
        kdf['prev_anom'] = kdf['is_anom'].shift(1, fill_value=False)
        # new segment starts where current is anomaly and (previous not anomaly or time gap != 1 minute)
        kdf['new_seg'] = kdf['is_anom'] & (~kdf['prev_anom'] | (kdf['ts_diff'] != pd.Timedelta(minutes=1)))
        # cumulative segment id
        kdf['seg_id'] = kdf['new_seg'].cumsum()
        # filter only anomaly rows
        anoms = kdf[kdf['is_anom']].copy()
        segments = []
        if not anoms.empty:
            grouped = anoms.groupby('seg_id')
            for _, g in grouped:
                start_ts = g['ts'].min()
                end_ts = g['ts'].max()
                length = int(g.shape[0])  # number of minutes in segment
                segments.append((start_ts.isoformat(), end_ts.isoformat(), length))
            # keep only first 5 segments
            segments = segments[:5]
    else:
        anomaly_count = 0
        max_value = np.nan
        earliest_anom_ts = pd.NaT
        breach_ratio = np.nan
        segments = []

    rows.append({
        'kpi_name': kpi,
        'p95': p95_val,
        'total_rows_in_window': total_rows,
        'anomaly_count_in_window': anomaly_count,
        'earliest_anomaly_timestamp_utc': earliest_anom_ts,
        'max_value_in_window': max_value,
        'breach_ratio': breach_ratio,
        'segments': segments
    })

result_orders = pd.DataFrame(rows)

# Sort rows by breach_ratio desc (NaNs last) then anomaly_count desc
result_orders = result_orders.sort_values(by=['breach_ratio', 'anomaly_count_in_window'], ascending=[False, False])

# Keep output compact: display only the requested columns
result_orders = result_orders[
    ['kpi_name', 'p95', 'total_rows_in_window', 'anomaly_count_in_window',
     'earliest_anomaly_timestamp_utc', 'max_value_in_window', 'breach_ratio', 'segments']
]

result_orders

```
Out[4]:
```
```python
# Summarize the orders KPI analysis from the previous result_orders DataFrame.
# Reuse result_orders computed earlier.

try:
    df = result_orders.copy()
except NameError:
    import pandas as pd
    df = pd.read_csv("metric.csv")  # fallback (should not be needed)
    df = df[df['cmdb_id']=='orders']  # minimal fallback

# Build concise summary lines using the existing dataframe values
lines = []
lines.append("Summary for cmdb_id='orders' (incident window 2024-01-19 03:47:24 — 2024-01-19 04:17:24 UTC):")
lines.append("")
for _, r in df.iterrows():
    k = r['kpi_name']
    p95 = r['p95']
    total = int(r['total_rows_in_window'])
    anom = int(r['anomaly_count_in_window'])
    earliest = r['earliest_anomaly_timestamp_utc']
    maxv = r['max_value_in_window']
    br = r['breach_ratio']
    # format values compactly
    p95_s = f"{p95:.6g}" if pd.notna(p95) else "NaN"
    max_s = f"{maxv:.6g}" if pd.notna(maxv) else "NaN"
    br_s = f"{br:.6g}" if pd.notna(br) else "NaN"
    earliest_s = str(earliest) if pd.notna(earliest) else "None"
    lines.append(f"- {k}: p95={p95_s}, rows={total}, anomalies={anom}, earliest={earliest_s}, max={max_s}, breach_ratio={br_s}")

lines.append("")
lines.append("Key observations:")
lines.append("- The latency KPIs show the largest relative breaches:")
lat50 = df[df['kpi_name']=='latency-50'].iloc[0]
lat90 = df[df['kpi_name']=='latency-90'].iloc[0]
lines.append(f"  * latency-50: p95={lat50['p95']:.6g}, anomalies={int(lat50['anomaly_count_in_window'])}, earliest={lat50['earliest_anomaly_timestamp_utc']}, breach_ratio={lat50['breach_ratio']:.6g} (largest).")
lines.append(f"  * latency-90: p95={lat90['p95']:.6g}, anomalies={int(lat90['anomaly_count_in_window'])}, earliest={lat90['earliest_anomaly_timestamp_utc']}, breach_ratio={lat90['breach_ratio']:.6g}.")
lines.append("- The 'error' KPI has p95=0.0 so all 25 in-window rows are flagged as anomalies (earliest 2024-01-19 03:50:00+00:00); this is driven by p95==0 behavior rather than large numeric increases.")
lines.append("- Other KPIs (socket, workload, cpu, mem) show only small relative breaches (breach_ratios << 0.1) with just 1-2 anomalous minutes each.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name           p95  total_rows_in_window  anomaly_count_in_window earliest_anomaly_timestamp_utc  max_value_in_window  breach_ratio                                           segments
2  latency-50  1.910247e-01                    24                        2      2024-01-19 03:52:00+00:00         1.452307e+00      6.602722  [(2024-01-19T03:52:00+00:00, 2024-01-19T03:52:...
3  latency-90  1.016053e+00                    24                        2      2024-01-19 03:52:00+00:00         3.155017e+00      2.105169  [(2024-01-19T03:52:00+00:00, 2024-01-19T03:53:...
5      socket  1.877667e+01                    25                        2      2024-01-19 03:55:00+00:00         1.958333e+01      0.042961  [(2024-01-19T03:55:00+00:00, 2024-01-19T03:55:...
6    workload  2.050013e+00                    25                        2      2024-01-19 04:06:00+00:00         2.082267e+00      0.015733  [(2024-01-19T04:06:00+00:00, 2024-01-19T04:06:...
0         cpu  1.027665e+00                    25                        2      2024-01-19 04:02:00+00:00         1.041572e+00      0.013533  [(2024-01-19T04:02:00+00:00, 2024-01-19T04:02:...
4         mem  3.312705e+08                    25                        2      2024-01-19 03:55:00+00:00         3.312788e+08      0.000025  [(2024-01-19T03:55:00+00:00, 2024-01-19T03:55:...
1       error  0.000000e+00                    25                       25      2024-01-19 03:50:00+00:00         0.000000e+00           NaN  [(2024-01-19T03:50:00+00:00, 2024-01-19T04:14:...```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse metric_df if available, else load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure timestamp parsed to UTC datetime column 'ts'
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Candidates and latency KPIs
candidates = ['carts', 'catalogue', 'orders', 'payment', 'user']
latency_kpis = ['latency-50', 'latency-90']

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full series (no time filtering)
p95_df = (
    metric_df[
        metric_df['cmdb_id'].isin(candidates) &
        metric_df['kpi_name'].isin(latency_kpis)
    ]
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95')
)

# Incident window (UTC)
start = pd.Timestamp("2024-01-19 03:47:24", tz='UTC')
end = pd.Timestamp("2024-01-19 04:17:24", tz='UTC')

# Rows in-window for the candidate latency KPIs
window_df = metric_df[
    (metric_df['cmdb_id'].isin(candidates)) &
    (metric_df['kpi_name'].isin(latency_kpis)) &
    (metric_df['ts'] >= start) &
    (metric_df['ts'] <= end)
].copy()

# Attach p95 to in-window rows for anomaly detection
window_with_p95 = pd.merge(
    window_df,
    p95_df,
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Mark anomalies (value >= p95) only where p95 is available
window_with_p95['is_anomaly'] = False
mask_valid = window_with_p95['p95'].notna() & window_with_p95['value'].notna()
window_with_p95.loc[mask_valid, 'is_anomaly'] = window_with_p95.loc[mask_valid, 'value'] >= window_with_p95.loc[mask_valid, 'p95']

# Aggregate in-window stats per group
agg = (
    window_with_p95
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        total_rows_in_window=('value', 'size'),
        anomaly_count_in_window=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp_utc=('ts', lambda s: s[window_with_p95.loc[s.index, 'is_anomaly']].min() if len(s)>0 else pd.NaT),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)

# 2) Merge global p95 with aggregated window stats to ensure all combos present in p95_df are included
result = pd.merge(p95_df, agg, on=['cmdb_id', 'kpi_name'], how='left')

# Fill defaults for combos with no in-window data
result['total_rows_in_window'] = result['total_rows_in_window'].fillna(0).astype(int)
result['anomaly_count_in_window'] = result['anomaly_count_in_window'].fillna(0).astype(int)
# earliest_anomaly_timestamp_utc and max_value_in_window remain NaT/NaN if absent

# 3) Compute breach_ratio = (max_value - p95)/p95, set NaN if p95==0 or max_value missing
result['breach_ratio'] = (result['max_value_in_window'] - result['p95']) / result['p95']
result.loc[(result['p95'] == 0) | (result['max_value_in_window'].isna()), 'breach_ratio'] = np.nan

# Ensure earliest_anomaly_timestamp_utc is timezone-aware UTC (if present)
result['earliest_anomaly_timestamp_utc'] = pd.to_datetime(result['earliest_anomaly_timestamp_utc']).dt.tz_convert('UTC')

# Select and order columns as requested
result_latency = result[
    ['cmdb_id', 'kpi_name', 'p95', 'total_rows_in_window', 'anomaly_count_in_window',
     'earliest_anomaly_timestamp_utc', 'max_value_in_window', 'breach_ratio']
].sort_values(by='breach_ratio', ascending=False)

# Keep output compact: top 20 rows (there will be at most 10 combos)
result_latency.head(20)

```
Out[5]:
```
Summary (latency KPIs for carts, catalogue, orders, payment, user during 2024-01-19 03:47:24 — 04:17:24 UTC):

Top groups by relative breach:
- orders / latency-50: p95 = 0.1910, rows = 24, anomalies = 2, earliest anomaly = 2024-01-19 03:52:00 UTC, max = 1.452307, breach_ratio ≈ 6.6027 (largest relative spike).
- orders / latency-90: p95 = 1.0161, rows = 24, anomalies = 2, earliest anomaly = 2024-01-19 03:52:00 UTC, max = 3.155017, breach_ratio ≈ 2.1052.

Other groups (smaller relative breaches):
- catalogue / latency-90: p95 = 0.004955, rows = 24, anomalies = 2, earliest = 2024-01-19 03:51:00 UTC, max = 0.00775, breach_ratio ≈ 0.5642.
- carts / latency-50: p95 = 0.80153, rows = 24, anomalies = 2, earliest = 2024-01-19 03:52:00 UTC, max = 1.244652, breach_ratio ≈ 0.5528.
- carts / latency-90: p95 = 3.760871, rows = 24, anomalies = 2, earliest = 2024-01-19 03:52:00 UTC, max = 5.386613, breach_ratio ≈ 0.4323.
- catalogue / latency-50: p95 = 0.00314, rows = 24, anomalies = 2, earliest = 2024-01-19 03:51:00 UTC, max = 0.003375, breach_ratio ≈ 0.0749.
- payment / latency-50: p95 = 0.002647, rows = 23, anomalies = 2, earliest = 2024-01-19 03:52:00 UTC, max = 0.002783, breach_ratio ≈ 0.0510.
- payment / latency-90: p95 = 0.004925, rows = 23, anomalies = 2, earliest = 2024-01-19 04:01:00 UTC, max = 0.005151, breach_ratio ≈ 0.0460.
- user / latency-50: p95 = 0.004863, rows = 24, anomalies = 2, earliest = 2024-01-19 04:12:00 UTC, max = 0.004909, breach_ratio ≈ 0.0093.
- user / latency-90: p95 = 0.009059, rows = 24, anomalies = 2, earliest = 2024-01-19 04:12:00 UTC, max = 0.009065, breach_ratio ≈ 0.0007.

Key takeaway:
- All candidate services had a small number of anomalous minutes (generally 2) for latency-50 and latency-90 during the window.
- orders shows the most significant relative latency spikes (especially latency-50), indicating orders experienced the largest departures from baseline latency during the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name       p95  total_rows_in_window  anomaly_count_in_window earliest_anomaly_timestamp_utc  max_value_in_window  breach_ratio
4     orders  latency-50  0.191025                    24                        2      2024-01-19 03:52:00+00:00             1.452307      6.602722
5     orders  latency-90  1.016053                    24                        2      2024-01-19 03:52:00+00:00             3.155017      2.105169
3  catalogue  latency-90  0.004955                    24                        2      2024-01-19 03:51:00+00:00             0.007750      0.564159
0      carts  latency-50  0.801530                    24                        2      2024-01-19 03:52:00+00:00             1.244652      0.552845
1      carts  latency-90  3.760871                    24                        2      2024-01-19 03:52:00+00:00             5.386613      0.432278
2  catalogue  latency-50  0.003140                    24                        2      2024-01-19 03:51:00+00:00             0.003375      0.074934
6    payment  latency-50  0.002647                    23                        2      2024-01-19 03:52:00+00:00             0.002783      0.051025
7    payment  latency-90  0.004925                    23                        2      2024-01-19 04:01:00+00:00             0.005151      0.046037
8       user  latency-50  0.004863                    24                        2      2024-01-19 04:12:00+00:00             0.004909      0.009303
9       user  latency-90  0.009059                    24                        2      2024-01-19 04:12:00+00:00             0.009065      0.000717```
```